In [7]:
import re
from paddleocr import TextRecognition

ocr = TextRecognition(
    model_name="th_PP-OCRv5_mobile_rec"
)

# ตัวที่ OCR ชอบสับสน -> เลข
NUM_MAP = {
    "O": "0",
    "Q": "0",
    "D": "0",
    "I": "1",
    "l": "1",
    "|": "1",

    "Z": "2",

    "S": "5",

    "B": "8",
}

# ไทยที่มักโดน OCR อ่านเป็นอังกฤษ
THAI_MAP = {
    "@": "ฮ",
    "&": "ฃ",
    "N": "ก",
    "n": "ก"
}

PLATE_PATTERN = re.compile(
    r'^[0-9]?[ก-ฮ]{1,3}[0-9]{1,4}$'
)


def clean_plate(text):

    # ลบช่องว่าง
    text = text.strip()

    # map อังกฤษ -> เลข
    for old, new in NUM_MAP.items():
        text = text.replace(old, new)

    # map พิเศษ
    for old, new in THAI_MAP.items():
        text = text.replace(old, new)

    # เก็บเฉพาะ ไทย อังกฤษ เลข
    text = re.sub(r'[^A-Za-zก-๙0-9]', '', text)

    # ลบอังกฤษที่เหลือ
    text = re.sub(r'[A-Za-z]', '', text)

    return text


results = ocr.predict("test_img/img2.jpg")

for res in results:

    if isinstance(res, dict):
        text = res.get("rec_text", "")
        score = res.get("rec_score", 0)
    else:
        text = str(res)
        score = 0

    plate = clean_plate(text)

    print("-" * 50)
    print(f"OCR Raw     : {text}")
    print(f"Confidence  : {score}")
    print(f"OCR Cleaned : {plate}")

    if PLATE_PATTERN.match(plate):
        print("VALID PLATE")
    else:
        print("INVALID PLATE")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kepin\.paddlex\official_models\th_PP-OCRv5_mobile_rec`.


--------------------------------------------------
OCR Raw     : จข 3888
Confidence  : 0.8515042066574097
OCR Cleaned : จข3888
VALID PLATE


In [8]:
import re
from pathlib import Path
from paddleocr import TextRecognition

ocr = TextRecognition(
    model_name="th_PP-OCRv5_mobile_rec"
)

# ตัวที่ OCR ชอบสับสน -> เลข
NUM_MAP = {
    "O": "0",
    "Q": "0",
    "D": "0",
    "I": "1",
    "l": "1",
    "|": "1",
    "Z": "2",
    "S": "5",
    "B": "8",
}

# ไทยที่มักโดน OCR อ่านเป็นอังกฤษ
THAI_MAP = {
    "@": "ฮ",
    "&": "ฃ",
    "N": "ก",
    "n": "ก"
}

PLATE_PATTERN = re.compile(
    r'^[0-9]?[ก-ฮ]{1,3}[0-9]{1,4}$'
)

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

FOLDER = r"D:\license_plate_and_car_detection\test_img"


def clean_plate(text):
    text = text.strip()
    for old, new in NUM_MAP.items():
        text = text.replace(old, new)
    for old, new in THAI_MAP.items():
        text = text.replace(old, new)
    text = re.sub(r'[^A-Za-zก-๙0-9]', '', text)
    text = re.sub(r'[A-Za-z]', '', text)
    return text


def parse_result(res):
    """รองรับทั้ง dict และ object ของ PaddleOCR v3"""
    if isinstance(res, dict):
        data = res.get("res", res)
        return data.get("rec_text", ""), data.get("rec_score", 0.0)
    if hasattr(res, "rec_text"):
        return res.rec_text, getattr(res, "rec_score", 0.0)
    if hasattr(res, "__dict__"):
        d = vars(res)
        return d.get("rec_text", ""), d.get("rec_score", 0.0)
    return str(res), 0.0


def process_image(image_path: str):
    """อ่านและแสดงผลภาพ 1 ใบ"""
    results = ocr.predict(image_path)
    for res in results:
        text, score = parse_result(res)
        plate = clean_plate(text)
        print(f"  OCR Raw    : {text!r}")
        print(f"  Confidence : {score:.3f}")
        print(f"  Cleaned    : {plate}")
        print(f"  Result     : {'✅ VALID' if PLATE_PATTERN.match(plate) else '❌ INVALID'}")
        print()


# ── Main ──────────────────────────────────
folder = Path(FOLDER)
image_files = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXT)

print(f"พบ {len(image_files)} ภาพใน {FOLDER}\n")

for i, img_path in enumerate(image_files, 1):
    print(f"{'='*50}")
    print(f"[{i}/{len(image_files)}] {img_path.name}")
    print(f"{'='*50}")
    try:
        process_image(str(img_path))
    except Exception as e:
        print(f"  ❌ Error: {e}\n")

Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kepin\.paddlex\official_models\th_PP-OCRv5_mobile_rec`.


พบ 10 ภาพใน D:\license_plate_and_car_detection\test_img

[1/10] img1.jpg
  OCR Raw    : 'นง 6734'
  Confidence : 0.963
  Cleaned    : นง6734
  Result     : ✅ VALID

[2/10] img10.png
  OCR Raw    : '30-6'
  Confidence : 0.482
  Cleaned    : 306
  Result     : ❌ INVALID

[3/10] img2.jpg
  OCR Raw    : 'จข 3888'
  Confidence : 0.852
  Cleaned    : จข3888
  Result     : ✅ VALID

[4/10] img3.jpg
  OCR Raw    : 'จจ 5564'
  Confidence : 0.761
  Cleaned    : จจ5564
  Result     : ✅ VALID

[5/10] img4.jpg
  OCR Raw    : '1ข84 5638'
  Confidence : 0.697
  Cleaned    : 1ข845638
  Result     : ❌ INVALID

[6/10] img5.jpg
  OCR Raw    : '5n0 3061'
  Confidence : 0.754
  Cleaned    : 5ก03061
  Result     : ❌ INVALID

[7/10] img6.jpg
  OCR Raw    : '2 65100'
  Confidence : 0.594
  Cleaned    : 265100
  Result     : ❌ INVALID

[8/10] img7.jpg
  OCR Raw    : '2ข 68155'
  Confidence : 0.663
  Cleaned    : 2ข68155
  Result     : ❌ INVALID

[9/10] img8.png
  OCR Raw    : 'n2 5773'
  Confidence : 0.729
  Cl

In [5]:
import re
import cv2
import numpy as np
from paddleocr import TextRecognition
from rapidfuzz import process, fuzz

# ─────────────────────────────────────────
# OCR Model
# ─────────────────────────────────────────
ocr = TextRecognition(model_name="th_PP-OCRv5_mobile_rec")

# ─────────────────────────────────────────
# Correction maps
# ─────────────────────────────────────────
# ตัวที่ OCR ชอบสับสน -> เลข
NUM_MAP = {
    "O": "0", "Q": "0", "D": "0",
    "I": "1", "l": "1", "|": "1",
    "Z": "2",
    "S": "5",
    "B": "8",
}

# อังกฤษ/พิเศษ -> ไทย (ใช้ก่อน NUM_MAP ในส่วนจังหวัด)
THAI_MAP = {
    "@": "ฮ",
    "&": "ฃ",
    "N": "ก",
    "n": "ก",
}

# ─────────────────────────────────────────
# Patterns
# ─────────────────────────────────────────
PLATE_PATTERN = re.compile(r'^[0-9]?[ก-ฮ]{1,3}[0-9]{1,4}$')

# ─────────────────────────────────────────
# รายชื่อจังหวัดทั้งหมด 77 จังหวัด
# ─────────────────────────────────────────
THAI_PROVINCES = [
    "กรุงเทพมหานคร", "กระบี่", "กาญจนบุรี", "กาฬสินธุ์", "กำแพงเพชร",
    "ขอนแก่น", "จันทบุรี", "ฉะเชิงเทรา", "ชลบุรี", "ชัยนาท",
    "ชัยภูมิ", "ชุมพร", "เชียงราย", "เชียงใหม่", "ตรัง",
    "ตราด", "ตาก", "นครนายก", "นครปฐม", "นครพนม",
    "นครราชสีมา", "นครศรีธรรมราช", "นครสวรรค์", "นนทบุรี", "นราธิวาส",
    "น่าน", "บึงกาฬ", "บุรีรัมย์", "ปทุมธานี", "ประจวบคีรีขันธ์",
    "ปราจีนบุรี", "ปัตตานี", "พระนครศรีอยุธยา", "พะเยา", "พังงา",
    "พัทลุง", "พิจิตร", "พิษณุโลก", "เพชรบุรี", "เพชรบูรณ์",
    "แพร่", "ภูเก็ต", "มหาสารคาม", "มุกดาหาร", "แม่ฮ่องสอน",
    "ยโสธร", "ยะลา", "ร้อยเอ็ด", "ระนอง", "ระยอง",
    "ราชบุรี", "ลพบุรี", "ลำปาง", "ลำพูน", "เลย",
    "ศรีสะเกษ", "สกลนคร", "สงขลา", "สตูล", "สมุทรปราการ",
    "สมุทรสงคราม", "สมุทรสาคร", "สระแก้ว", "สระบุรี", "สิงห์บุรี",
    "สุโขทัย", "สุพรรณบุรี", "สุราษฎร์ธานี", "สุรินทร์", "หนองคาย",
    "หนองบัวลำภู", "อ่างทอง", "อำนาจเจริญ", "อุดรธานี", "อุตรดิตถ์",
    "อุทัยธานี", "อุบลราชธานี",
]

# ─────────────────────────────────────────
# Helper: แปลง result object -> (text, score)
# ─────────────────────────────────────────
def parse_ocr_result(res):
    """รองรับทั้ง dict และ object แบบ PaddleOCR v3"""
    if isinstance(res, dict):
        data = res.get("res", res)
        return data.get("rec_text", ""), data.get("rec_score", 0.0)
    # object ที่มี attribute
    if hasattr(res, "rec_text"):
        return res.rec_text, getattr(res, "rec_score", 0.0)
    # fallback: แปลงเป็น dict ผ่าน __dict__
    if hasattr(res, "__dict__"):
        d = vars(res)
        return d.get("rec_text", ""), d.get("rec_score", 0.0)
    return str(res), 0.0


# ─────────────────────────────────────────
# Clean: ส่วนทะเบียน (บน)
# ─────────────────────────────────────────
def clean_plate_number(text: str) -> str:
    text = text.strip()
    for old, new in NUM_MAP.items():
        text = text.replace(old, new)
    for old, new in THAI_MAP.items():
        text = text.replace(old, new)
    # เก็บเฉพาะ ไทย และ เลข
    text = re.sub(r'[^ก-๙0-9]', '', text)
    return text


# ─────────────────────────────────────────
# Clean: ส่วนจังหวัด (ล่าง)
# ─────────────────────────────────────────
def clean_province_text(text: str) -> str:
    text = text.strip()
    for old, new in THAI_MAP.items():
        text = text.replace(old, new)
    # เก็บเฉพาะ ไทย (จังหวัดไม่มีเลข)
    text = re.sub(r'[^ก-๙]', '', text)
    return text


# ─────────────────────────────────────────
# Fuzzy match จังหวัด
# ─────────────────────────────────────────
def match_province(text: str, threshold: int = 70):
    """
    คืน (matched_province, score) หรือ (None, 0) ถ้าไม่ผ่าน threshold
    ใช้ token_set_ratio เพื่อรับมือกับข้อความสั้น/ตัดออก
    """
    if not text:
        return None, 0
    result = process.extractOne(
        text,
        THAI_PROVINCES,
        scorer=fuzz.token_set_ratio,
    )
    if result is None:
        return None, 0
    match, score, _ = result
    if score >= threshold:
        return match, score
    return None, score


# ─────────────────────────────────────────
# Split ภาพป้ายทะเบียนเป็นบน/ล่าง
# ─────────────────────────────────────────
def split_plate_image(image: np.ndarray, split_ratio: float = 0.60):
    """
    แบ่งภาพออกเป็น 2 ส่วน
    - บน  (split_ratio)      : ตัวอักษร + เลขทะเบียน
    - ล่าง (1 - split_ratio)  : ชื่อจังหวัด

    split_ratio=0.60 หมายถึง 60% บน 40% ล่าง
    ปรับได้ตามขนาดป้ายจริง
    """
    h, w = image.shape[:2]
    cut = int(h * split_ratio)
    top = image[:cut, :]
    bottom = image[cut:, :]
    return top, bottom


# ─────────────────────────────────────────
# OCR helper: ส่งภาพ numpy array เข้า model
# ─────────────────────────────────────────
def ocr_image(image: np.ndarray):
    """คืน list ของ (text, score)"""
    results = ocr.predict(image)
    output = []
    for res in results:
        text, score = parse_ocr_result(res)
        output.append((text, score))
    return output


# ─────────────────────────────────────────
# Main pipeline: อ่านป้ายทะเบียน 1 ภาพ
# ─────────────────────────────────────────
def read_plate(image_path: str, split_ratio: float = 0.60,
               province_threshold: int = 70, debug: bool = True):
    """
    อ่านป้ายทะเบียนจากภาพ 1 ใบ
    คืน dict:
        plate_number  : str  เลขทะเบียนที่ clean แล้ว
        province_raw  : str  ข้อความดิบจาก OCR ส่วนล่าง
        province      : str | None  จังหวัดที่ match ได้ (None ถ้าไม่ผ่าน threshold)
        province_score: int  คะแนน fuzzy 0-100
        valid_plate   : bool ผ่าน regex หรือเปล่า
        top_raw       : list ข้อความดิบจากส่วนบน
        bottom_raw    : list ข้อความดิบจากส่วนล่าง
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"ไม่พบไฟล์: {image_path}")

    top_img, bottom_img = split_plate_image(img, split_ratio)

    # ── อ่านส่วนบน (ทะเบียน) ──
    top_results = ocr_image(top_img)
    top_texts = [r[0] for r in top_results]
    plate_raw = " ".join(top_texts)
    plate_number = clean_plate_number(plate_raw)

    # ── อ่านส่วนล่าง (จังหวัด) ──
    bottom_results = ocr_image(bottom_img)
    bottom_texts = [r[0] for r in bottom_results]
    province_raw_joined = " ".join(bottom_texts)
    province_cleaned = clean_province_text(province_raw_joined)
    province_match, province_score = match_province(
        province_cleaned, threshold=province_threshold
    )

    valid = bool(PLATE_PATTERN.match(plate_number))

    result = {
        "plate_number": plate_number,
        "province_raw": province_raw_joined,
        "province": province_match,
        "province_score": province_score,
        "valid_plate": valid,
        "top_raw": top_results,
        "bottom_raw": bottom_results,
    }

    if debug:
        _print_result(result)

    return result


def _print_result(r: dict):
    sep = "=" * 55
    print(sep)
    print("📋  ส่วนบน (ทะเบียน)")
    for text, score in r["top_raw"]:
        print(f"   OCR Raw    : {text!r}  (conf={score:.3f})")
    print(f"   → Cleaned  : {r['plate_number']}")
    print(f"   → Valid    : {'✅ VALID' if r['valid_plate'] else '❌ INVALID'}")
    print()
    print("🗺️  ส่วนล่าง (จังหวัด)")
    for text, score in r["bottom_raw"]:
        print(f"   OCR Raw    : {text!r}  (conf={score:.3f})")
    print(f"   → Cleaned  : {r['province_raw']}")
    if r["province"]:
        print(f"   → Province : {r['province']}  (fuzzy={r['province_score']})")
    else:
        print(f"   → Province : ไม่พบจังหวัด  (fuzzy={r['province_score']})")
    print(sep)


# ─────────────────────────────────────────
# Batch: อ่านทุกภาพในโฟลเดอร์
# ─────────────────────────────────────────
def read_folder(folder_path: str, split_ratio: float = 0.60,
                province_threshold: int = 70, debug: bool = True) -> list[dict]:
    """
    อ่านป้ายทะเบียนจากทุกภาพในโฟลเดอร์
    รองรับ .jpg .jpeg .png .bmp .webp
    คืน list ของ result dict (เพิ่ม key 'file')
    """
    from pathlib import Path

    folder = Path(folder_path)
    if not folder.exists():
        raise FileNotFoundError(f"ไม่พบโฟลเดอร์: {folder_path}")

    IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_files = sorted([
        p for p in folder.iterdir()
        if p.suffix.lower() in IMAGE_EXT
    ])

    if not image_files:
        print(f"⚠️  ไม่พบไฟล์ภาพใน {folder_path}")
        return []

    print(f"🔍  พบ {len(image_files)} ภาพใน {folder_path}\n")

    all_results = []
    for i, img_path in enumerate(image_files, 1):
        print(f"[{i}/{len(image_files)}] {img_path.name}")
        try:
            result = read_plate(
                str(img_path),
                split_ratio=split_ratio,
                province_threshold=province_threshold,
                debug=debug,
            )
            result["file"] = img_path.name
            all_results.append(result)
        except Exception as e:
            print(f"   ❌ Error: {e}")
            all_results.append({
                "file": img_path.name,
                "plate_number": "",
                "province": None,
                "province_score": 0,
                "valid_plate": False,
                "error": str(e),
            })

    # ── สรุปผล ──
    print("\n" + "=" * 55)
    print(f"{'ไฟล์':<30} {'ทะเบียน':<12} {'จังหวัด':<18} {'Valid'}")
    print("-" * 55)
    for r in all_results:
        plate  = r.get("plate_number", "ERROR")
        prov   = r.get("province") or f"? ({r.get('province_score', 0)})"
        valid  = "✅" if r.get("valid_plate") else "❌"
        print(f"{r['file']:<30} {plate:<12} {prov:<18} {valid}")
    print("=" * 55)

    return all_results


# ─────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────
if __name__ == "__main__":
    # ── กำหนด path โฟลเดอร์ตรงนี้เลย (ไม่ใช้ sys.argv เพราะ Jupyter ขัดกัน) ──
    FOLDER = r"D:\license_plate_and_car_detection\test_img"

    read_folder(FOLDER, split_ratio=0.60, province_threshold=70, debug=True)


Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\kepin\.paddlex\official_models\th_PP-OCRv5_mobile_rec`.


🔍  พบ 10 ภาพใน D:\license_plate_and_car_detection\test_img

[1/10] img1.jpg
📋  ส่วนบน (ทะเบียน)
   OCR Raw    : 'นง 6734'  (conf=0.987)
   → Cleaned  : นง6734
   → Valid    : ✅ VALID

🗺️  ส่วนล่าง (จังหวัด)
   OCR Raw    : '- ซลบุริ -'  (conf=0.698)
   → Cleaned  : - ซลบุริ -
   → Province : ไม่พบจังหวัด  (fuzzy=66.66666666666666)
[2/10] img10.png
📋  ส่วนบน (ทะเบียน)
   OCR Raw    : '30-663'  (conf=0.892)
   → Cleaned  : 30663
   → Valid    : ❌ INVALID

🗺️  ส่วนล่าง (จังหวัด)
   OCR Raw    : ''  (conf=0.000)
   → Cleaned  : 
   → Province : ไม่พบจังหวัด  (fuzzy=0)
[3/10] img2.jpg
📋  ส่วนบน (ทะเบียน)
   OCR Raw    : 'จข 3888'  (conf=0.981)
   → Cleaned  : จข3888
   → Valid    : ✅ VALID

🗺️  ส่วนล่าง (จังหวัด)
   OCR Raw    : 'ขลบุรี'  (conf=0.780)
   → Cleaned  : ขลบุรี
   → Province : ชลบุรี  (fuzzy=83.33333333333333)
[4/10] img3.jpg
📋  ส่วนบน (ทะเบียน)
   OCR Raw    : 'จจ5564'  (conf=0.998)
   → Cleaned  : จจ5564
   → Valid    : ✅ VALID

🗺️  ส่วนล่าง (จังหวัด)
   OCR Raw    : 'ชลมุรี9